# Dataset Inventory Analysis

This notebook performs a scan of the raw datasets stored in the project's data directory. It extracts metadata from each Excel spreadsheet (including sheet names, row/column counts, and file sizes) to compile a centralized dataset inventory report.

In [1]:
# Install required libraries in the active Jupyter kernel
%pip install pandas numpy openpyxl jinja2
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup and Environment Configuration

We import the necessary libraries (`pandas`, `numpy`, `pathlib.Path`, and `openpyxl`) and configure standard settings for DataFrame visualization. We also locate the project's root and raw data directory.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl

# Set pandas options for optimal DataFrame display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

# Define project root and raw data directory
# Notebook is in project/notebooks/00_dataset_inventory.ipynb, so we need .parent to reach the root
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Path: {RAW_DATA}")


Project Root: c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform
Raw Data Path: c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw


## 2. Verify Data Directory Path

We verify that the raw data directory path actually exists.

In [3]:
RAW_DATA.exists()


True

## 3. Find Raw Dataset Files

We search for and list all Excel files (`.xlsx`) in the raw data directory.

In [4]:
# Use pathlib to find all .xlsx files
excel_files = sorted(RAW_DATA.glob("*.xlsx"))

print(f"Datasets Found : {len(excel_files)}")

for file in excel_files:
    print(file.name)


Datasets Found : 7
analysis.xlsx
balancesheet.xlsx
cashflow.xlsx
companies.xlsx
documents.xlsx
profitandloss.xlsx
prosandcons.xlsx


## 4. Extract Metadata from Excel Files

We iterate through all found Excel files, use `openpyxl` to extract sheet names, read the sheets into DataFrames to calculate rows and columns, and compute file sizes to compile a comprehensive metadata dataset.

In [5]:
import os

inventory = []

# Compile the inventory by reading the actual data and sizes
for file in excel_files:
    try:
        # Calculate file size in KB
        file_size_kb = file.stat().st_size / 1024
        
        # Read the excel file using pandas to easily get rows/columns
        # sheet_name=None reads all sheets into a dictionary of DataFrames
        excel_data = pd.read_excel(file, sheet_name=None)
        
        sheets = list(excel_data.keys())
        total_rows = 0
        max_columns = 0
        
        # Aggregate rows and columns across all sheets in the workbook
        for sheet_name, df in excel_data.items():
            total_rows += len(df)
            max_columns = max(max_columns, len(df.columns))
            
        # Append all metadata
        inventory.append({
            'Dataset Name': file.name,
            'Sheets': ", ".join(sheets),
            'Rows': total_rows,
            'Columns': max_columns,
            'File Size (KB)': file_size_kb,
            'File Path': str(file)
        })
        
    except Exception as e:
        print(f"Could not read {file.name}: {e}")

# Create DataFrame
inventory_df = pd.DataFrame(inventory)
inventory_df.head()


Could not read balancesheet.xlsx: [Errno 13] Permission denied: 'c:\\Users\\polai\\OneDrive\\Desktop\\N100 FINANCIAL INTELLIGENCE PLATFORM\\nifty100-financial-intelligence-platform\\data\\raw\\balancesheet.xlsx'


,Dataset Name,Sheets,Rows,Columns,File Size (KB),File Path
0,analysis.xlsx,Analysis,21,6,9.973633,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
1,cashflow.xlsx,Cash Flow,1188,7,52.612305,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
2,companies.xlsx,Companies,93,12,25.461914,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
3,documents.xlsx,Documents,1586,4,49.404297,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
4,profitandloss.xlsx,Profit & Loss,1277,15,104.833984,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...


## 5. Sort the Inventory

We sort the compiled dataset inventory alphabetically by the name of the dataset.

In [6]:
# Sort alphabetically by the dataset name column
if not inventory_df.empty:
    inventory_df = inventory_df.sort_values(by="Dataset Name").reset_index(drop=True)

inventory_df


,Dataset Name,Sheets,Rows,Columns,File Size (KB),File Path
0,analysis.xlsx,Analysis,21,6,9.973633,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
1,cashflow.xlsx,Cash Flow,1188,7,52.612305,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
2,companies.xlsx,Companies,93,12,25.461914,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
3,documents.xlsx,Documents,1586,4,49.404297,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
4,profitandloss.xlsx,Profit & Loss,1277,15,104.833984,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...
5,prosandcons.xlsx,Pros & Cons,17,4,6.099609,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL...


## 6. Save the Report to CSV

We save the sorted dataset inventory DataFrame as a CSV file in the project's reports directory.

In [7]:
# Create reports directory if it doesn't exist
REPORT_PATH = PROJECT_ROOT / "reports"
REPORT_PATH.mkdir(exist_ok=True)

# Save the dataframe to a CSV file
if not inventory_df.empty:
    inventory_df.to_csv(REPORT_PATH / "dataset_inventory.csv", index=False)
    print("Dataset Inventory Saved Successfully")
else:
    print("Inventory is empty. No CSV was saved.")


Dataset Inventory Saved Successfully


## 7. Compute Summary Metrics

We calculate and display aggregate metrics across all datasets, including the total number of datasets, total rows, total columns, and total storage size.

In [8]:
print("Total Datasets")
display(len(inventory_df))

print("Total Rows")
display(inventory_df["Rows"].sum() if not inventory_df.empty else 0)

print("Total Columns")
display(inventory_df["Columns"].sum() if not inventory_df.empty else 0)

print("Total Storage (KB)")
display(round(inventory_df["File Size (KB)"].sum(), 2) if not inventory_df.empty else 0)


Total Datasets


6

Total Rows


np.int64(4182)

Total Columns


np.int64(48)

Total Storage (KB)


np.float64(248.39)

## 8. Style the Inventory Table

We apply gradient highlights using pandas styling features for better readability.

In [9]:
if not inventory_df.empty:
    # Highlight numerical values with a blue gradient
    display(inventory_df.style.background_gradient(cmap="Blues", subset=['Rows', 'Columns', 'File Size (KB)']))
else:
    print("No data to display.")


,Dataset Name,Sheets,Rows,Columns,File Size (KB),File Path
0,analysis.xlsx,Analysis,21,6,9.973633,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw\analysis.xlsx
1,cashflow.xlsx,Cash Flow,1188,7,52.612305,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw\cashflow.xlsx
2,companies.xlsx,Companies,93,12,25.461914,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw\companies.xlsx
3,documents.xlsx,Documents,1586,4,49.404297,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw\documents.xlsx
4,profitandloss.xlsx,Profit & Loss,1277,15,104.833984,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw\profitandloss.xlsx
5,prosandcons.xlsx,Pros & Cons,17,4,6.099609,c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform\data\raw\prosandcons.xlsx


## 9. Final Summary

*The final summary will be updated here after executing the notebook.*